# Deep Neural Networks

**IMSE 774 · Neural Networks · Week 4 · Sessions 7–8 · September 15 & 17, 2026**

Every code cell from the [lecture note](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/04-deep-networks.html), in the order it appears
there. The prose, figures, and exercises live on that page; this notebook is
for running and changing the code.

Run a cell with **Shift+Enter**. Run the setup cell first — the later cells
depend on the names it defines.

Source: [https://github.com/harunpirim/IMSE774-NeuralNetworks](https://github.com/harunpirim/IMSE774-NeuralNetworks)


## Setup


`setup`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from math import comb

torch.manual_seed(774)
np.random.seed(774)

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

NDSU_GREEN = "#005643"
NDSU_GOLD  = "#FFC72C"
ACCENT     = "#C1121F"

## Composing Two Networks


`fig-compose`


In [ ]:
def shallow(x, theta, phi0, phi):
    """Equations (4.1)-(4.2): a shallow ReLU network at every x. Returns y and the pre-activations."""
    pre = theta[:, 0:1] + theta[:, 1:2] * x[None, :]
    return phi0 + phi @ np.maximum(pre, 0), pre

def count_regions(pre_list):
    """Number of distinct activation patterns across every hidden unit of every layer."""
    patterns = np.concatenate([p > 0 for p in pre_list], axis=0).T
    return len(np.unique(patterns, axis=0))

# Network 1: zigzag on [-1, 1] with slopes 3, -3, 3 and joints at -1/3 and 1/3.
theta_A = np.array([[1.0, 1.0], [1/3, 1.0], [-1/3, 1.0]])
phi0_A, phi_A = -1.0, np.array([3.0, -6.0, 6.0])
# Network 2: a three-region function of y on [-1, 1], joints at -1/2 and 0.3.
theta_B = np.array([[1.0, 1.0], [0.5, 1.0], [-0.3, 1.0]])
phi0_B, phi_B = 0.6, np.array([-2.0, 3.5, -2.5])

x  = np.linspace(-1, 1, 20001)[1:-1]
y,  pre_A = shallow(x, theta_A, phi0_A, phi_A)
yp, pre_B = shallow(y, theta_B, phi0_B, phi_B)
yB_direct, _ = shallow(x, theta_B, phi0_B, phi_B)   # network 2 on its own input axis

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
axes[0].plot(x, y, color=NDSU_GREEN, lw=2.2)
axes[0].set_title("Network 1: $y[x]$"); axes[0].set_xlabel("$x$"); axes[0].set_ylabel("$y$")
axes[1].plot(x, yB_direct, color=ACCENT, lw=2.2)
axes[1].set_title("Network 2: $y'[y]$"); axes[1].set_xlabel("$y$"); axes[1].set_ylabel("$y'$")
axes[2].plot(x, yp, color="k", lw=2.2)
axes[2].set_title("Composition: $y'[y[x]]$"); axes[2].set_xlabel("$x$"); axes[2].set_ylabel("$y'$")
for ax in axes:
    ax.set_xlim(-1, 1); ax.set_ylim(-1.05, 1.05)
for j in (-1/3, 1/3):
    axes[0].axvline(j, color="gray", ls=":", lw=1); axes[2].axvline(j, color="gray", ls=":", lw=1)
plt.tight_layout()
plt.show()

print(f"network 1 alone : {count_regions([pre_A]):>2} regions from 3 hidden units")
print(f"network 2 alone : {count_regions([shallow(x, theta_B, phi0_B, phi_B)[1]]):>2} regions from 3 hidden units")
print(f"composition     : {count_regions([pre_A, pre_B]):>2} regions from 6 hidden units")
print(f"shallow, 6 units: at most 7")

## Folding the input space


`fig-fold-depth`


In [ ]:
def compose_K(x, K):
    y, pres = x.copy(), []
    for _ in range(K):
        y, pre = shallow(y, theta_A, phi0_A, phi_A)
        pres.append(pre)
    return y, pres

fig, axes = plt.subplots(1, 4, figsize=(12, 2.8), sharey=True)
rows = []
for ax, K in zip(axes, [1, 2, 3, 4]):
    yK, pres = compose_K(x, K)
    n_units, n_par = 3 * K, 3 * 2 + (K - 1) * 3 * 4 + 4
    regions = count_regions(pres)
    ax.plot(x, yK, color=NDSU_GREEN, lw=1.6)
    ax.set_title(f"$K = {K}$: {regions} regions"); ax.set_xlabel("$x$")
    rows.append((K, n_units, n_par, regions, 3 ** K, n_units + 1))
axes[0].set_ylabel("$y$")
plt.tight_layout()
plt.show()

print(f"{'layers K':>9}{'hidden units':>14}{'parameters':>12}{'regions':>9}{'3^K':>6}{'shallow max':>13}")
for K, n_units, n_par, regions, cube, shallow_max in rows:
    print(f"{K:>9}{n_units:>14}{n_par:>12}{regions:>9}{cube:>6}{shallow_max:>13}")

## From Composition to a Deep Network


`outer-product-check`


In [ ]:
theta1, phi1_0, phi1 = np.array([[-0.2, 0.4], [-0.9, 0.9], [1.1, -0.7]]), -0.23, np.array([-1.3, 1.3, 0.66])
theta2, phi2_0, phi2 = np.array([[0.3, -1.5], [-0.4, 2.0], [0.6, 1.2]]), 0.10, np.array([0.9, -0.7, 0.5])

xs = np.linspace(0, 2, 9)
y_mid, _  = shallow(xs, theta1, phi1_0, phi1)          # first network
y_comp, _ = shallow(y_mid, theta2, phi2_0, phi2)       # ... fed into the second

Psi  = np.outer(theta2[:, 1], phi1)                    # equation (4.6): psi_ij = theta'_i1 phi_j
psi0 = theta2[:, 0] + theta2[:, 1] * phi1_0            # psi_i0 = theta'_i0 + theta'_i1 phi_0
H1 = np.maximum(theta1[:, 0:1] + theta1[:, 1:2] * xs, 0)
H2 = np.maximum(psi0[:, None] + Psi @ H1, 0)
y_deep = phi2_0 + phi2 @ H2                            # two-layer network, equations (4.7)-(4.9)

print(f"max |composition - two-layer network| = {np.abs(y_comp - y_deep).max():.1e}")
print(f"rank of the outer-product Psi         = {np.linalg.matrix_rank(Psi)}   (a free 3x3 Psi has rank up to 3)")

## Deep Neural Networks


`fig-clipping`


In [ ]:
theta = np.array([[-0.2, 0.4], [-0.9, 0.9], [1.1, -0.7]])      # layer 1: Prince figure 3.3
psi0  = np.array([0.41, -0.45, -0.54])                          # layer 2 biases
Psi   = np.array([[ 1.1, -1.4, -0.9],
                  [-1.0,  1.3,  1.8],
                  [ 1.8,  0.0,  0.6]])                          # layer 2 weights
phi_p0, phi_p = 0.1, np.array([1.0, -0.6, 0.9])                 # output layer

xc = np.linspace(0, 2, 20001)[1:-1]
pre1 = theta[:, 0:1] + theta[:, 1:2] * xc
H1   = np.maximum(pre1, 0)
pre2 = psi0[:, None] + Psi @ H1
H2   = np.maximum(pre2, 0)
W2   = phi_p[:, None] * H2
out  = phi_p0 + W2.sum(axis=0)

joints1 = -theta[:, 0] / theta[:, 1]
joints2 = [xc[np.where(np.diff(np.sign(pre2[i])) != 0)[0]] for i in range(3)]
colors  = [ACCENT, "#1864AB", NDSU_GREEN]

fig = plt.figure(figsize=(9.5, 9))
gs = fig.add_gridspec(4, 3)
rows = [(pre2, r"$\psi_{%d0} + \psi_{%d1}h_1 + \psi_{%d2}h_2 + \psi_{%d3}h_3$"),
        (H2,   r"$h'_%d$ (clipped)"),
        (W2,   r"$\phi'_%d h'_%d$")]
for r, (data, title) in enumerate(rows):
    for d in range(3):
        ax = fig.add_subplot(gs[r, d])
        ax.plot(xc, data[d], color=colors[d], lw=2)
        ax.axhline(0, color="k", lw=0.6)
        for j in joints1:
            ax.axvline(j, color="gray", ls=":", lw=0.9)
        for j in joints2[d]:
            ax.axvline(j, color=NDSU_GOLD, lw=1.4)
        n = d + 1
        ax.set_title(title % ((n,) * title.count("%d")))
        ax.set_ylim(-1.7, 1.7)
        if r == 2:
            ax.set_xlabel("$x$")
ax = fig.add_subplot(gs[3, 1])
ax.plot(xc, out, color="k", lw=2.4)
for j in joints1:
    ax.axvline(j, color="gray", ls=":", lw=0.9)
for js in joints2:
    for j in js:
        ax.axvline(j, color=NDSU_GOLD, lw=1.4)
ax.set_title(r"$y' = \phi'_0 + \phi'_1 h'_1 + \phi'_2 h'_2 + \phi'_3 h'_3$"); ax.set_xlabel("$x$")
plt.tight_layout()
plt.show()

n_new = sum(len(js) for js in joints2)
print(f"joints from layer 1            : {len(joints1)}")
print(f"new joints added by layer 2    : {n_new}  ({', '.join(str(len(js)) for js in joints2)} per unit)")
print(f"linear regions in the output   : {count_regions([pre1, pre2])}  (= 1 + {len(joints1)} + {n_new})")
print(f"shallow network with 6 units   : at most 7")

## The general formulation


`matrix-shapes`


In [ ]:
def make_mlp(D_i, widths, D_o):
    """A fully connected ReLU network: equation (4.15) with hidden widths D_1, ..., D_K."""
    layers, d = [], D_i
    for w in widths:
        layers += [nn.Linear(d, w), nn.ReLU()]
        d = w
    layers.append(nn.Linear(d, D_o))
    return nn.Sequential(*layers)

net = make_mlp(D_i=3, widths=[4, 2, 3], D_o=2)     # figure 4.6
linears = [m for m in net if isinstance(m, nn.Linear)]
print(f"{'layer':<8}{'Omega_k shape':>16}{'beta_k size':>13}   reads")
for k, lin in enumerate(linears):
    src = "x" if k == 0 else f"h_{k}"
    dst = "y" if k == len(linears) - 1 else f"h_{k + 1}"
    print(f"k = {k:<4}{str(tuple(lin.weight.shape)):>16}{lin.bias.numel():>13}   {dst} = a[beta_{k} + Omega_{k} {src}]" if dst != "y"
          else f"k = {k:<4}{str(tuple(lin.weight.shape)):>16}{lin.bias.numel():>13}   {dst} = beta_{k} + Omega_{k} {src}")
print(f"\ntotal parameters: {sum(p.numel() for p in net.parameters())}")

## Linear regions per parameter


`fig-regions-depth`


In [ ]:
def n_params(D, K, D_i=1, D_o=1):
    D, K = int(D), int(K)
    return D * (D_i + 1) + (K - 1) * D * (D + 1) + D_o * (D + 1)

def n_regions(D, K, D_i=1):
    D, K = int(D), int(K)                      # Python ints: these counts overflow int64
    if D_i == 1:
        return (D + 1) ** K
    return (D // D_i + 1) ** (D_i * (K - 1)) * sum(comb(D, j) for j in range(D_i + 1))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
shades = plt.cm.viridis(np.linspace(0.85, 0.1, 5))
for K, c in zip(range(1, 6), shades):
    Ds = np.arange(3, 19)
    axes[0].plot([n_params(D, K) for D in Ds], [n_regions(D, K) for D in Ds], "o-", ms=3, color=c, lw=1.8, label=f"$K = {K}$")
    Ds = np.arange(10, 101, 10)
    axes[1].plot([n_params(D, K, D_i=10) for D in Ds], [float(n_regions(D, K, D_i=10)) for D in Ds], "o-", ms=3, color=c, lw=1.8, label=f"$K = {K}$")
axes[0].plot(n_params(10, 5), n_regions(10, 5), "o", color="k", ms=8, zorder=5)
axes[1].plot(n_params(50, 5, D_i=10), float(n_regions(50, 5, D_i=10)), "o", color="k", ms=8, zorder=5)
axes[0].set_title("$D_i = 1$ input"); axes[1].set_title("$D_i = 10$ inputs")
for ax in axes:
    ax.set_yscale("log"); ax.set_xlabel("number of parameters"); ax.set_ylabel("maximum linear regions")
axes[0].set_xlim(0, 1000); axes[1].set_xlim(0, 20000)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"K = 5, D = 10, D_i = 1 : {n_params(10, 5):>6,} parameters, {n_regions(10, 5):>10,} regions")
print(f"K = 5, D = 50, D_i = 10: {n_params(50, 5, D_i=10):>6,} parameters, about 10^{np.log10(float(n_regions(50, 5, D_i=10))):.0f} regions")

## Large, structured inputs


`dense-image`


In [ ]:
pixels = 1000 * 1000 * 3                       # one RGB photograph
for D1 in (100, 1000):
    p = D1 * (pixels + 1)
    print(f"fully connected first layer, {D1:>5} hidden units: {p:>14,} parameters  ({p * 4 / 1e9:5.1f} GB in float32)")

## Deep Networks in PyTorch


`shallow-vs-deep`


In [ ]:
def triangle(x):                                   # period 0.5, amplitude 1
    return 2 * np.abs(2 * ((2 * x) % 1) - 1) - 1

x_train = np.random.uniform(-1, 1, 400)
y_train = triangle(x_train) + np.random.normal(0, 0.05, x_train.shape)
X = torch.tensor(x_train, dtype=torch.float32)[:, None]
Y = torch.tensor(y_train, dtype=torch.float32)[:, None]
xg = torch.linspace(-1, 1, 20001)[1:-1][:, None]

def realized_regions(model, xg):
    """Distinct activation patterns over the grid, across every ReLU in the network."""
    pats, h = [], xg
    with torch.no_grad():
        for layer in model:
            h = layer(h)
            if isinstance(layer, nn.ReLU):
                pats.append(h > 0)
    return len(np.unique(torch.cat(pats, dim=1).numpy(), axis=0))

def fit(model, epochs=5000, lr=0.01):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for _ in range(epochs):
        opt.zero_grad()
        loss = nn.MSELoss()(model(X), Y)
        loss.backward(); opt.step()
        history.append(loss.item())
    return history

torch.manual_seed(774); shallow_net = make_mlp(1, [95], 1)
torch.manual_seed(774); deep_net    = make_mlp(1, [5] * 10, 1)
hist_s = fit(shallow_net)
hist_d = fit(deep_net)

print(f"{'network':<22}{'parameters':>11}{'max regions':>13}{'realized':>10}{'final MSE':>11}")
for name, m, hist, mx in [("shallow, D = 95", shallow_net, hist_s, n_regions(95, 1)),
                          ("deep, K = 10, D = 5", deep_net, hist_d, n_regions(5, 10))]:
    print(f"{name:<22}{sum(p.numel() for p in m.parameters()):>11}{mx:>13,}{realized_regions(m, xg):>10}{hist[-1]:>11.4f}")

`fig-shallow-vs-deep`


In [ ]:
with torch.no_grad():
    xg_np = xg.squeeze().numpy()
    ys = shallow_net(xg).squeeze().numpy()
    yd = deep_net(xg).squeeze().numpy()

fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.3))
for ax, yy, title in [(axes[0], ys, "shallow: $D = 95$"), (axes[1], yd, "deep: $K = 10$, $D = 5$")]:
    ax.scatter(x_train, y_train, s=6, color="gray", alpha=0.4)
    ax.plot(xg_np, triangle(xg_np), "--", color=ACCENT, lw=1.2, label="target")
    ax.plot(xg_np, yy, color=NDSU_GREEN, lw=2, label="network")
    ax.set_title(title); ax.set_xlabel("$x$"); ax.set_ylim(-1.4, 1.4)
axes[0].set_ylabel("$y$"); axes[0].legend(fontsize=8, loc="upper right")
axes[2].plot(hist_s, color=NDSU_GREEN, lw=1.4, label="shallow")
axes[2].plot(hist_d, color=ACCENT, lw=1.4, label="deep")
axes[2].set_yscale("log"); axes[2].set_xlabel("epoch"); axes[2].set_ylabel("MSE"); axes[2].set_title("Training loss"); axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()